
# N13: Tabular Foundation Models (Bagged TabPFN)
**Objective:** Execute Rule 17 (The Novelty Directive). This notebook abandons Gradient Boosted Decision Trees in favor of a 2026 State-of-the-Art Tabular Foundation Model (TabPFN).

**The Architectural Challenge (For Future Model Reviewers):**
TabPFN (Prior-Data Fitted Networks) utilizes a Transformer architecture to perform in-context zero-shot tabular learning. Due to the quadratic scaling of self-attention, TabPFN has strict context limits (typically 10,000 rows maximum per forward pass). Because our training dataset is 690,088 rows, we cannot natively fit TabPFN to the entire dataset at once. 

**The Solution:**
We deploy a **Massive Bagged TabPFN Meta-Ensemble**.
1. We preprocess and quantile-transform all features (Neural Networks require scaled inputs).
2. For each CV fold, we randomly sample $M$ non-overlapping strata of 5,000 to 10,000 rows.
3. We initialize $M$ separate TabPFN context windows.
4. We aggregate the zero-shot probability distributions across all $M$ models to generate our OOF and Test predictions.


In [ ]:

# Set BEFORE pip/import so a cold kernel never triggers license download.
import os
os.environ["TABPFN_MODEL_CACHE_DIR"] = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
os.environ["TABPFN_NO_BROWSER"] = "1"
!pip install -q tabpfn


In [ ]:

# MUST set cache dir BEFORE importing tabpfn (settings are read at import time).
import os
_KAGGLE_TABPFN = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
TABPFN_CKPT = f"{_KAGGLE_TABPFN}/tabpfn-v3-classifier-v3_default.ckpt"
os.environ["TABPFN_MODEL_CACHE_DIR"] = _KAGGLE_TABPFN
os.environ["TABPFN_NO_BROWSER"] = "1"

if not os.path.isfile(TABPFN_CKPT):
    raise FileNotFoundError(
        f"Missing mounted checkpoint:\n  {TABPFN_CKPT}\n"
        "Add notebook input: Models → prior-labsai/tabpfn-3 (default).\n"
        "Then Factory reset runtime and Run All from the top."
    )
print(f"TabPFN checkpoint OK: {TABPFN_CKPT}")

import pandas as pd
import numpy as np
import warnings
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder, QuantileTransformer
from sklearn.impute import SimpleImputer
from tabpfn import TabPFNClassifier

warnings.filterwarnings('ignore')

# Global Config
ID_COL, TARGET_COL = "id", "health_condition"
CLASSES = ("at-risk", "fit", "unhealthy")
N_FOLDS = 5
SEED = 2026
GPU_ENABLED = torch.cuda.is_available()
DEVICE = 'cuda' if GPU_ENABLED else 'cpu'

# TabPFN Constraints
TABPFN_CONTEXT_LIMIT = 5000  # Max rows per TabPFN instance
N_BAGS_PER_FOLD = 10         # How many TabPFN models to ensemble per fold

print(f"Executing on: {DEVICE} | Ensembling {N_BAGS_PER_FOLD} TabPFNs per fold.")


In [ ]:

# 1. Load Data
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')

# Target Encoding and Preprocessing
cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality']
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

le_target = LabelEncoder()
y_train = le_target.fit_transform(train_df[TARGET_COL])
X_train_raw = train_df[cat_cols + num_cols].copy()
X_test_raw = test_df[cat_cols + num_cols].copy()

for col in cat_cols:
    X_train_raw[col] = X_train_raw[col].fillna('Missing').astype(str)
    X_test_raw[col] = X_test_raw[col].fillna('Missing').astype(str)

num_imputer = SimpleImputer(strategy='median')
X_train_num_raw = num_imputer.fit_transform(train_df[num_cols])
X_test_num_raw = num_imputer.transform(test_df[num_cols])


In [ ]:

# 2. Advanced Neural Preprocessing (Quantile Transformation)
def get_te_matrices(tr_i, va_i):
    X_tr_cat, X_va_cat = X_train_raw[cat_cols].iloc[tr_i], X_train_raw[cat_cols].iloc[va_i]
    X_tr_num, X_va_num = X_train_num_raw[tr_i], X_train_num_raw[va_i]
    y_tr = y_train[tr_i]
    
    fold_te_tr, fold_te_val, fold_te_test = [], [], []
    for col in cat_cols:
        crosstab = pd.crosstab(X_tr_cat[col], y_tr, normalize='index')
        mapping = crosstab.to_dict('index')
        mean_mapping = crosstab.mean().to_dict()
        
        tr_mapped = X_tr_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_tr.append(pd.DataFrame(tr_mapped.tolist()).values)
        va_mapped = X_va_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_val.append(pd.DataFrame(va_mapped.tolist()).values)
        te_mapped = X_test_raw[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_test.append(pd.DataFrame(te_mapped.tolist()).values)
        
    X_tr_full = np.hstack([X_tr_num, np.hstack(fold_te_tr)])
    X_va_full = np.hstack([X_va_num, np.hstack(fold_te_val)])
    X_te_full = np.hstack([X_test_num_raw, np.hstack(fold_te_test)])
    
    # Quantile Transform for Neural Architectures
    qt = QuantileTransformer(output_distribution='normal', random_state=SEED)
    X_tr_full = qt.fit_transform(X_tr_full)
    X_va_full = qt.transform(X_va_full)
    X_te_full = qt.transform(X_te_full)
    
    return X_tr_full, X_va_full, X_te_full


In [ ]:

# 3. TabPFN Bagged Meta-Ensemble Training Loop
oof_probs = np.zeros((len(train_df), 3))
test_probs = np.zeros((len(test_df), 3))
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_i, va_i) in enumerate(skf.split(X_train_raw, y_train)):
    print(f"--- FOLD {fold+1}/{N_FOLDS} ---")
    X_tr_full, X_va_full, X_te_full = get_te_matrices(tr_i, va_i)
    y_tr_full = y_train[tr_i]
    
    fold_va_preds = np.zeros((len(va_i), 3))
    fold_te_preds = np.zeros((len(test_df), 3))
    
    # Stratified Sampling to build non-overlapping context bags
    bag_indices = np.random.permutation(len(X_tr_full))
    
    for bag in range(N_BAGS_PER_FOLD):
        # Extract subset for this bag
        start_idx = bag * TABPFN_CONTEXT_LIMIT
        end_idx = start_idx + TABPFN_CONTEXT_LIMIT
        if end_idx > len(X_tr_full): break
        
        subset_idx = bag_indices[start_idx:end_idx]
        X_bag = X_tr_full[subset_idx]
        y_bag = y_tr_full[subset_idx]
        
        # Initialize and Fit TabPFN Context
        # Pass absolute model_path so TabPFN never attempts a gated HF download.
        # n_estimators=1 because we macro-ensemble bags ourselves.
        clf = TabPFNClassifier(
            device=DEVICE,
            n_estimators=1,
            random_state=SEED + bag,
            model_path=TABPFN_CKPT,
        )
        clf.fit(X_bag, y_bag)
        
        # Aggregate Probabilities
        fold_va_preds += clf.predict_proba(X_va_full) / N_BAGS_PER_FOLD
        fold_te_preds += clf.predict_proba(X_te_full) / N_BAGS_PER_FOLD
        print(f"  Bag {bag+1}/{N_BAGS_PER_FOLD} contextualized and inferred.")
        
    oof_probs[va_i] = fold_va_preds
    test_probs += fold_te_preds / N_FOLDS
    
    fold_score = balanced_accuracy_score(y_train[va_i], fold_va_preds.argmax(axis=1))
    print(f"Fold {fold+1} BAcc: {fold_score:.5f}\n")

final_oof_score = balanced_accuracy_score(y_train, oof_probs.argmax(axis=1))
print(f"=== TabPFN Meta-Ensemble Final OOF BAcc: {final_oof_score:.5f} ===")


In [ ]:

# 4. Generate Final TFM Submission
final_preds = test_probs.argmax(axis=1)

sub_df = pd.DataFrame({
    ID_COL: test_df[ID_COL].astype("int64"),
    TARGET_COL: [CLASSES[i] for i in final_preds]
})
sub_df.to_csv("submission.csv", index=False)
print("Exported TabPFN submission.csv")
